In [ ]:
### import libraries

# for DL modeling
import torch
import torch.nn as nn
from torch.utils.data import DataLoader,TensorDataset
from sklearn.model_selection import train_test_split

# for number-crunching
import numpy as np
import scipy.stats as stats

# for dataset management
import pandas as pd

# for data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Import and process the data

In [ ]:
url = "data.csv"

data = pd.read_csv(url,sep=';')
data

In [ ]:
# describe the data
data.describe()

This pandas code prints the **number of unique values** in each column of the wine quality dataset.


```
for i in data.keys():
```


**Loops over column names**:

- `data.keys()`: returns Index object `['fixed acidity', 'volatile 

acidity', ..., 'quality', 'boolQuality']`

- `i`: each column name (string) on iteration

```
print(f'{i} has {len(np.unique(data[i]))} unique values')
```

**Prints unique count per column** using f-string formatting:

- `i`: column name (e.g., `'alcohol'`)

- `data[i]`: pandas Series for that column (e.g., ~1590 alcohol values)

- `np.unique(data[i])`: array of distinct values in column

- `len(...)`: count of unique values






In [ ]:
# list number of unique values per column

for i in data.keys():
  print(f'{i} has {len(np.unique(data[i]))} unique values')

This seaborn code creates a **pairplot visualization** of wine quality 

features with regression lines colored by quality score.


```
cols2plot = ['fixed acidity','volatile acidity','citric acid','quality']
```

**Selects 4 columns** for pairwise plotting from wine dataset:

- 3 continuous features + quality (target)

```

sns.pairplot(data[cols2plot],kind='reg',hue='quality')
```

**Creates pairplot grid** (4×4 subplots):

- `data[cols2plot]`: subset DataFrame with 4 columns

- `kind='reg'`: **off-diagonal**: scatterplot + **linear regression 

line** per quality group

- `hue='quality'`: colors points/lines by quality score (3,4,5,6,7,8)

- **Diagonal**: histograms of each variable (colored by quality)

```
plt.show()
```

Displays the complete pairplot figure.



In [ ]:
# pairwise plots
cols2plot = ['fixed acidity','volatile acidity','citric acid','quality']
sns.pairplot(data[cols2plot],kind='reg',hue='quality')
plt.show()

This code creates a **boxplot visualization** of all wine quality features and then **removes outliers**.


```
fig,ax = plt.subplots(1,figsize=(17,4))
```

**Creates wide single subplot**:

- `plt.subplots(1,...)`: 1 row figure (single plot)

- `figsize=(17,4)`: **wide rectangle** (17" wide, 4" tall) to fit 12+ 

boxplots horizontally

```
ax = sns.boxplot(data=data)
```

**Boxplot for every numeric column**:

- `sns.boxplot(data=data)`: automatically plots **all numeric columns** 

side-by-side

- X-axis: column names (fixed acidity, volatile acidity, ..., 

boolQuality)

- Y-axis: values for each feature

- Each box shows: median (line), IQR (box), whiskers (1.5×IQR), 

outliers (dots)

```
ax.set_xticklabels(ax.get_xticklabels(),rotation=45)
```

**Rotates x-axis labels**:

- `ax.get_xticklabels()`: gets current column name labels

- `rotation=45`: tilts 45° **counterclockwise** for readability

- Prevents long feature names from overlapping

```
plt.show()
```

Displays boxplot **(visualize outliers first)**.

```
data = data[data['total sulfur dioxide']<200]
```

**Removes outliers**:

- `data['total sulfur dioxide']<200`: boolean mask (~few rows True for 

≥200)

- `data[...]` : **filters rows** keeping only sulfur dioxide < 200

- **Drops extreme values** identified visually from boxplot

- Modifies `data` in-place (fewer rows now)



In [ ]:
# plot some data
fig,ax = plt.subplots(1,figsize=(17,4))
ax = sns.boxplot(data=data)
ax.set_xticklabels(ax.get_xticklabels(),rotation=45)
plt.show()


#remove rows with outliers
data = data[data['total sulfur dioxide']<200]

In [ ]:
### z-score all variables except for quality

# find the columns we want to normalize (all except quality)
cols2zscore = data.keys()
cols2zscore = cols2zscore.drop('quality')

# z-score (written out for clarity)
for col in cols2zscore:
  meanval   = np.mean(data[col])
  stdev     = np.std(data[col],ddof=1)
  data[col] = (data[col]-meanval) / stdev

# can also do more compactly
#data[cols2zscore] = data[cols2zscore].apply(stats.zscore)

data.describe()

In [ ]:
# check the plot again
fig,ax = plt.subplots(1,figsize=(17,4))
ax = sns.boxplot(data=data)
ax.set_xticklabels(ax.get_xticklabels(),rotation=45)
plt.show()

In [ ]:
# distribution quality values
fig = plt.figure(figsize=(10,7))
plt.rcParams.update({'font.size': 22}) # increase font size in the figure

counts = data['quality'].value_counts()
plt.bar(list(counts.keys()),counts)
plt.xlabel('Quality rating')
plt.ylabel('Count')
plt.show()

# create a new column for binarized (boolean) quality
data['boolQuality'] = 0
# data['boolQuality'][data['quality']<6] = 0 # implicit in the code! just here for clarity
data['boolQuality'][data['quality']>5] = 1

data[['quality','boolQuality']]

# Re-organize the data: train/test in DataLoaders

In [ ]:
# convert from pandas dataframe to tensor
dataT  = torch.tensor( data[cols2zscore].values ).float()
labels = torch.tensor( data['boolQuality'].values ).float()

print( dataT.shape )
print( labels.shape )

# we'll actually need the labels to be a "tensor"
labels = labels[:,None]
print( labels.shape )

## splitting data into training and testing 

This code splits the  dataset into train/test sets and creates PyTorch DataLoaders for batching during neural network training.



```

train_data,test_data, train_labels,

test_labels = train_test_split(data, labels, 

test_size=.2)
```

Splits tensors using scikit-learn:

- `train_test_split(X, y, test_size=0.2)`: 

80/20 train/test split (320 train, 80 test 

samples)

- `data`: feature tensor `(400, 2)`

- `labels`: label tensor `(400, 1)`

- Returns 4 tensors: `train_data`(320×2), 

`test_data`(80×2), `train_labels`(320×1), 

`test_labels`(80×1)

- Syntax: Unpacks into 4 variables via tuple 

assignment

```
train_data = TensorDataset(train_data,

train_labels)
```

Wraps train tensors into `torch.utils.data.

TensorDataset`:

- Creates dataset object pairing features/

labels by index

- Now `train_data[i]` returns tuple `

(feature_tensor_i, label_tensor_i)`

- Enables indexed access for DataLoader

```
test_data  = TensorDataset(test_data,

test_labels)
```

Same for test set

```
batchsize    = 16 #int(train_data.tensors[0].

shape[0]/4) -- Hard-coding is better to 

avoid huge batches!

```

Sets batch size to 16:

- Train: 320/16 = 20 batches

- Commented alternative `train_data.tensors

[0].shape[0]` gets 320 samples, `/4` = 80 

(too large)

- Fixed 16 prevents memory issues with small 

datasets

```

train_loader = DataLoader(train_data,

batch_size=batchsize,shuffle=True)

```

Creates training DataLoader:

- `DataLoader(dataset, batch_size=16, 

shuffle=True)`: yields 20 batches of 16 

samples

- `shuffle=True`: randomizes order each 

epoch (prevents overfitting to sequence)

- Each batch: `(features_batch: 16×2, 

labels_batch: 16×1)`

- Syntax: `for X_batch, y_batch in 

train_loader: ...`

```

test_loader  = DataLoader(test_data,

batch_size=test_data.tensors[0].shape[0])
```

Creates test DataLoader:

- `test_data.tensors[0].shape[0]` = 80 (full 

test set size)

- `batch_size=80`: single batch containing 

ALL test samples

- `shuffle=False` (default): deterministic 

evaluation

- Yields 1 batch: `(test_features: 80×2, 

test_labels: 80×1)`



In [ ]:
# use scikitlearn to split the data
train_data,test_data, train_labels,test_labels = train_test_split(dataT, labels, test_size=.1)


# then convert them into PyTorch Datasets (note: already converted to tensors)
train_data = TensorDataset(train_data,train_labels)
test_data  = TensorDataset(test_data,test_labels)


# finally, translate into dataloader objects
batchsize    = 64
train_loader = DataLoader(train_data,batch_size=batchsize,shuffle=True)
test_loader  = DataLoader(test_data,batch_size=test_data.tensors[0].shape[0])

In [ ]:

for X,y in train_loader:
  print(X.shape,y.shape)



In [ ]:
import torch.nn.functional as F

This PyTorch class implements a **batch 

normalization (BN) ablation network** for 

wine quality prediction, allowing toggling 

BN on/off with one parameter.

## Class Structure
```
class ANNwine_withBNorm(nn.Module):
```

Inherits `nn.Module` - PyTorch neural 

network base class.

## Constructor Layers
```
def __init__(self):

  super().__init__()
```

Initializes module (no custom parameters).

```

self.input = nn.Linear(11,16)
```

**Input layer**: 11 wine features → 16 

hidden units.

```
self.fc1 = nn.Linear(16,32)

self.bnorm1 = nn.BatchNorm1d(16)
```

**Layer 1**:

- `fc1`: 16→32 linear transformation

- `bnorm1`: **BatchNorm1d(16)** normalizes 

**16 input features** (matches `fc1` input 

size)

```

self.fc2 = nn.Linear(32,20)

self.bnorm2 = nn.BatchNorm1d(32)
```

**Layer 2**:

- `fc2`: 32→20 linear transformation

- `bnorm2`: **BatchNorm1d(32)** normalizes 

**32 input features**

```

self.output = nn.Linear(20,1)
```

**Output layer**: 20→1 (binary wine quality 

logit).


## Forward Pass (Key Feature)
```

def forward(self,x,doBN):
```

**Toggle parameter** `doBN` (True/False) 

switches BN behavior.

```

x = F.relu( self.input(x) )
```

**Input processing**: Linear → ReLU (input 

already z-scored, no BN needed).


## BN Path (`if doBN`)
```

# Layer 1


x = self.bnorm1(x)  # Normalize batch: 

mean=0, std=1 per feature

x = self.fc1(x)     # Linear weighted 

combination

x = F.relu(x)       # ReLU nonlinearity

# Layer 2  

x = self.bnorm2(x)  # Normalize

x = self.fc2(x)     # Linear

x = F.relu(x)       # ReLU
```

**Order**: **BN → Linear → ReLU** (standard 

practice).


## No-BN Path (`else`)

```

x = F.relu( self.fc1(x) )  # Linear → ReLU

x = F.relu( self.fc2(x) )  # Linear → ReLU
```

**BN disabled**: standard feedforward 

without normalization.

```
return self.output(x)
```

**Final prediction**: `(batch_size, 1)` 

logits.

## Architecture Diagram
```

Input (11) → Linear→ReLU → [BN(16)

→Linear→ReLU → BN(32)→Linear→ReLU] → Output

(1)
                                 ↑ Toggle with doBN
```



In [ ]:
# create a class for the model WITH BATCH NORM

class ANNwine_withBNorm(nn.Module):
  def __init__(self):
    super().__init__()

    ### input layer
    self.input = nn.Linear(11,16)
    
    ### hidden layers
    self.fc1    = nn.Linear(16,32)
    self.bnorm1 = nn.BatchNorm1d(16) # the number of units into this layer
    self.fc2    = nn.Linear(32,20)
    self.bnorm2 = nn.BatchNorm1d(32) # the number of units into this layer

    ### output layer
    self.output = nn.Linear(20,1)
  
  # forward pass
  def forward(self,x,doBN):

    # input (x starts off normalized)
    x = F.relu( self.input(x) )


    if doBN:
      # hidden layer 1
      x = self.bnorm1(x) # batchnorm
      x = self.fc1(x)    # weighted combination
      x = F.relu(x)      # activation function

      # hidden layer 2
      x = self.bnorm2(x) # batchnorm
      x = self.fc2(x)    # weighted combination
      x = F.relu(x)      # activation function
    

    else:
      # hidden layer 1
      x = F.relu( self.fc1(x) )

      # hidden layer 2
      x = F.relu( self.fc2(x) )

    # output layer
    return self.output(x)

## TRAINING AND TESTING

This PyTorch function trains the 

**batch-normalized wine quality model** with 

toggleable BN, tracking full learning curves 

over 1000 epochs.

### Function Setup
```
def trainTheModel(doBN=True):
```

Training function with **BN toggle** 

parameter (default: enabled).

```

lossfun = nn.BCEWithLogitsLoss()

optimizer = torch.optim.SGD(winenet.

parameters(),lr=.01)
```

**Binary classification setup**:

- `BCEWithLogitsLoss()`: combines sigmoid + 

BCE (numerically stable for logits)

- `SGD(lr=0.01)`: simple gradient descent on 

model parameters

```

losses   = torch.zeros(numepochs)

trainAcc = []

testAcc  = []
```

**Metric storage**:

- `losses`: tensor for epoch losses 

(pre-allocated)

- `trainAcc`, `testAcc`: lists for epoch 

accuracies


### Training Loop
```

for epochi in range(numepochs):
```

Loops over 1000 epochs.

```

winenet.train()
```

**Activates training mode**:

- Enables dropout, batchnorm updates running 

stats

- Required before training batches

### Batch Training
```

for X,y in train_loader:
```

Iterates over training batches (~44 batches 

of 32 samples).

```

yHat = winenet(X,doBN)

loss = lossfun(yHat,y)
```

**Forward pass**:

- `winenet(X,doBN)`: model prediction with 

BN toggle

- `yHat.shape = (32,1)` logits

- `lossfun`: BCE loss between logits and 

binary labels

```

optimizer.zero_grad()

loss.backward()

optimizer.step()

```

**Standard backpropagation step**.

```

batchLoss.append(loss.item())

batchAcc.append( 100*torch.mean(((yHat>0) == 

y).float()).item() )

```

**Batch metrics**:

- `yHat>0`: threshold logits (>0 predicts 

class 1)

- `(predicted == y)`: boolean accuracy mask

- `torch.mean(...).item()`: converts to 

Python float → `*100` for %

### Epoch-End Aggregation
```

trainAcc.append( np.mean(batchAcc) )

losses[epochi] = np.mean(batchLoss)
```

**Averages batch metrics** for epoch 

summaries.

### Test Evaluation

```

winenet.eval()

X,y = next(iter(test_loader))

```

**Switches to eval mode** + extracts full 

test batch (~159 samples).

```

with torch.no_grad():

  yHat = winenet(X,doBN)

```

**Inference** (no gradients, same BN toggle):

- Uses running BN stats (not batch stats)

- Memory efficient

```

testAcc.append( 100*torch.mean(((yHat>0) == 

y).float()).item() )

```

**Test accuracy** using same threshold logic.


### Return Value
```

return trainAcc,testAcc,losses

```

Returns epoch-wise histories for plotting/

analysis.

## Key Design

```

BN=True:  Faster convergence, higher final 

accuracy

BN=False: Baseline (slower, prone to 

internal covariate shift)
```


**Perfect ablation**: Train once, compare 

learning curves! `doBN` enables fair A/B 

testing.

In [ ]:
# a function that trains the model

# global parameter
numepochs = 1000

def trainTheModel(doBN=True):

  # loss function and optimizer
  lossfun = nn.BCEWithLogitsLoss()
  optimizer = torch.optim.SGD(winenet.parameters(),lr=.01)

  # initialize losses
  losses   = torch.zeros(numepochs)
  trainAcc = []
  testAcc  = []

  # loop over epochs
  for epochi in range(numepochs):

    # switch on training mode
    winenet.train()

    # loop over training data batches
    batchAcc  = []
    batchLoss = []
    for X,y in train_loader:

      # forward pass and loss
      yHat = winenet(X,doBN)
      loss = lossfun(yHat,y)

      # backprop
      optimizer.zero_grad()
      loss.backward()
      optimizer.step()

      # loss from this batch
      batchLoss.append(loss.item())

      # compute training accuracy for this batch
      batchAcc.append( 100*torch.mean(((yHat>0) == y).float()).item() )
    # end of batch loop...

    # now that we've trained through the batches, get their average training accuracy
    trainAcc.append( np.mean(batchAcc) )

    # and get average losses across the batches
    losses[epochi] = np.mean(batchLoss)



    ### test accuracy
    winenet.eval()
    X,y = next(iter(test_loader)) # extract X,y from test dataloader
    with torch.no_grad(): # deactivates autograd
      yHat = winenet(X,doBN)
    testAcc.append( 100*torch.mean(((yHat>0) == y).float()).item() )
  
  # function output
  return trainAcc,testAcc,losses

In [ ]:
winenet = ANNwine_withBNorm()
trainAccNo,testAccNo,lossesNo = trainTheModel(False)

# create and train a model WITH BATCHNORM
winenet = ANNwine_withBNorm()
trainAccWith,testAccWith,lossesWith = trainTheModel(True)


In [ ]:
# plot the results
fig,ax = plt.subplots(1,3,figsize=(17,5))

ax[0].plot(lossesWith,label='WITH batchnorm')
ax[0].plot(lossesNo,label='NO batchnorm')
ax[0].set_title('Losses')
ax[0].legend()

ax[1].plot(trainAccWith,label='WITH batchnorm')
ax[1].plot(trainAccNo,label='NO batchnorm')
ax[1].set_title('Train accuracy')
ax[1].legend()

ax[2].plot(testAccWith,label='WITH batchnorm')
ax[2].plot(testAccNo,label='NO batchnorm')
ax[2].set_title('Test accuracy')
ax[2].legend()

plt.show()

# Additional explorations

In [ ]:
# 1) In a later video, we will use DL to predict residual sugar. Use seaborn to make a histogram of that data column.
#    Spend a minute to explore the visualization options in sns.histplot. For example, you can add a kernel density 
#    estimate, make the histogram bars purple, and so on.
# 
# 2) (Warning: This exercise is for people who are familiar with statistics.) Loop over all the variables in the dataset,
#    and perform an independent-samples t-test on the data for the binarized wine quality. Which variables are significantly
#    different between "low" and "high" quality wine?
# 